# GAIM Executive App - Complete Data Layer Reference

All tables live in `datagroup_mdl.mdl_sales_analytics` on Databricks Unity Catalog.

Use Databricks SQL warehouse or SDK with app service principal:
- `324a6ec7-e988-42c7-8a7f-55465f5bea37`

## 1. Primary Output Tables (Written by `forecast_unify_publish`)

### `forecast_results` and `forecast_results_latest` (view = latest `run_date`)
- 12,527 rows
- Partitioned by `product`
- Overwritten daily

| Column | Type | Description |
|---|---|---|
| run_date | DATE | Publish date (today run) |
| ds | DATE | Forecast week (Monday) |
| product | STRING | UCC, ITSG, Total |
| sales_market | STRING | Total, NA, EMEA, APAC, LATAM, Unknown |
| grain_level | STRING | total or market |
| model | STRING | Model name or ACTUAL |
| forecast_type | STRING | actuals, rolling, roy, forward_1yr |
| forecast_view | STRING | adaptive (blends actuals) or boq (pure model) |
| actual | DOUBLE | Weekly ARR actual (only when forecast_type='actuals') |
| p10 | DOUBLE | 10th percentile scenario |
| p50 | DOUBLE | Median scenario |
| p90 | DOUBLE | 90th percentile scenario |
| most_likely | DOUBLE | Equals p50 |
| recommended_for_exec | BIGINT | 1 if model=Adaptive_Ensemble else 0 |
| source_run_ts | TIMESTAMP | Upstream notebook run timestamp |
| published_at_utc | TIMESTAMP | Row write time |
| model_wape | DOUBLE | Holdout WAPE from monitoring baseline |
| model_mape | DOUBLE | Holdout MAPE (ITSG only; UCC NULL) |

Models:
- ACTUAL
- Adaptive_Ensemble
- DHR_ARIMA
- ETS
- Global_LGB_Q50_ITSG
- Global_LGB_Q50_UCC
- MSTL_v2
- Prophet_trend

`forecast_type` logic (disjoint):
- `actuals`: closed weeks with known ARR (`model=ACTUAL`, `forecast_view=adaptive` only)
- `rolling`: next 13 weeks (current quarter forward)
- `roy`: rest of fiscal year after rolling window (through 2026-12-28)
- `forward_1yr`: weeks beyond FY end (2027+)

### `forecast_insights`
- 3 rows
- Overwritten daily

| Column | Type | Description |
|---|---|---|
| run_date | DATE | Publish date |
| product | STRING | UCC, ITSG, or scope value |
| scope | STRING | Product scope label |
| insights_json | STRING | Full JSON payload (headline, bullets, stats) |
| model_confidence | DOUBLE | 0-100 confidence score |
| momentum | STRING | ACCELERATING, DECELERATING, STABLE |
| risk_level | STRING | LOW RISK, MODERATE RISK, HIGH RISK |
| published_at_utc | TIMESTAMP | Write timestamp |

### `forecast_graph_explanations`
- 300 rows (3 products x 5 regions x 2 periods x 10 graphs)

| Column | Type | Description |
|---|---|---|
| run_date | DATE | Publish date |
| product | STRING | UCC, ITSG, Total |
| region | STRING | Total, NA, EMEA, APAC, LATAM |
| scope_period | STRING | quarter or year |
| graph_key | STRING | weekly, ytd, seasonality, by_product, monthly, accuracy, model_lab, backtest, quarterly_outlook, multi_model |
| title | STRING | Human-readable chart title |
| explanation | STRING | Multi-paragraph explanation text |
| bullets | STRING | JSON array of highlights |
| generated_at | TIMESTAMP | Generation time |

Compatibility views:
- `forecast_results_v2compat`: wide drop-in replacement for legacy `arr_forecast_v2` schema
- `forecast_leaderboard_v2compat`: per-model accuracy leaderboard by product
- `forecast_insights_v2compat`: insights reshaped for legacy app endpoint

## 2. UCC Upstream Planning Tables

### `ucc_forecast_v5`
- 756 rows
- 21 forward weeks
- 6 models x (21 total + 105 market) = 756

| Column Group | Type | Description |
|---|---|---|
| model | STRING | Model name |
| ds | DATE | Forecast week |
| forecast | DOUBLE | Point forecast (= p50) |
| p10-p90 (9 cols) | DOUBLE | Full quantile fan by 10s |
| grain_level | STRING | total or market |
| sales_market | STRING | Market label |
| quarter | STRING | Example: 2026Q3 |
| fiscal_year | STRING | Example: FY2026 |
| source | STRING | Source identifier |
| run_date_utc | DATE | Run date |
| run_timestamp_utc | TIMESTAMP | Run timestamp |
| recommended_for_exec | BIGINT | 1 for Adaptive_Ensemble |
| notebook_name/version | STRING | Source notebook metadata |
| product_group | STRING | UCC |
| pipeline_version | STRING | Example: V5 |

### `ucc_forecast_boq`
- 444 rows
- Rolling 1-year BOQ plan
- Forecast only
- Full p10 to p90 quantiles (increase of 10)

### `ucc_forecast_cbe`
- 426 rows
- Current Best Estimate = actuals (closed weeks) + forecast (open weeks)
- Includes `cumulative_qtd`, `boq_weekly`, `boq_cumulative_qtd`, `pace_vs_boq_pct`

### `ucc_forecast_vintages`
- 17,424 rows
- Append-only forecast history for vintage drift tracking

### `ucc_forecast_monitoring_baseline`
- 439 rows
- Per-model baselines and alert thresholds (WAPE/sMAPE/RMSE)

### `ucc_forecast_v4`
- 792 rows
- Legacy mirror (same schema as V5)

## 3. ITSG Upstream Planning Tables

### `itsg_forecast_v5`
- 972 rows
- 27 forward weeks
- 6 models x (27 total + 135 market) = 972
- Quantiles: p10, p50, p90 only

| Column Group | Type | Description |
|---|---|---|
| ds | TIMESTAMP | Forecast week |
| model | STRING | Model name |
| grain_level | STRING | total or market |
| sales_market | STRING | Market label |
| forecast | DOUBLE | Point forecast (= p50) |
| p10, p50, p90 | DOUBLE | Quantiles |
| product_group | STRING | ITSG |
| run_timestamp_utc | TIMESTAMP | Run timestamp |
| run_date_utc | TIMESTAMP | Run date |
| notebook_name/version | STRING | Source notebook metadata |
| recommended_for_exec | BIGINT | 1 for Adaptive_Ensemble |

Note: ITSG uses p10/p50/p90 only (no intermediate quantiles).

### `itsg_forecast_boq`
- 432 rows
- Rolling 1-year BOQ plan
- Forecast only, p10/p50/p90

### `itsg_forecast_cbe`
- 432 rows
- Current Best Estimate = actuals (closed weeks) + forecast (open weeks)
- Includes `cumulative_qtd`, `boq_weekly`, `boq_cumulative_qtd`, `pace_vs_boq_pct`

### `itsg_forecast_monitoring_baseline`
- 409 rows
- Per-model baselines and evaluation metrics

### `itsg_forecast_v4`
- 972 rows
- Legacy mirror (same schema as `itsg_forecast_v5`)

## 4. Reading Patterns

### Pattern A - Overview KPI (Current Quarter Landing)
```sql
SELECT product,
       SUM(CASE WHEN forecast_type = 'actuals' THEN actual ELSE 0 END) AS actual_qtd,
       SUM(CASE WHEN forecast_type = 'rolling' AND model = 'Adaptive_Ensemble'
           THEN p50 ELSE 0 END) AS forecast_remaining,
       SUM(CASE WHEN forecast_type = 'actuals' THEN actual ELSE 0 END)
       + SUM(CASE WHEN forecast_type = 'rolling' AND model = 'Adaptive_Ensemble'
           THEN p50 ELSE 0 END) AS quarter_landing
FROM datagroup_mdl.mdl_sales_analytics.forecast_results_latest
WHERE forecast_view = 'adaptive'
  AND grain_level = 'total'
GROUP BY product;
```

### Pattern B - Weekly Forecast Chart (Adaptive)
```sql
SELECT ds, product, actual, p10, p50, p90, forecast_type
FROM datagroup_mdl.mdl_sales_analytics.forecast_results_latest
WHERE forecast_view = 'adaptive'
  AND model IN ('ACTUAL', 'Adaptive_Ensemble')
  AND grain_level = 'total'
ORDER BY product, ds;
```

### Pattern C - BOQ Plan Target (Stable Reference)
```sql
SELECT ds, product, p10, p50, p90, forecast_type
FROM datagroup_mdl.mdl_sales_analytics.forecast_results_latest
WHERE forecast_view = 'boq'
  AND model = 'Adaptive_Ensemble'
  AND grain_level = 'total'
ORDER BY product, ds;
```

### Pattern D - Pace vs Plan (CBE tables)
```sql
-- UCC
SELECT ds, model, forecast, source, cumulative_qtd, boq_weekly,
       boq_cumulative_qtd, pace_vs_boq_pct, quarter
FROM datagroup_mdl.mdl_sales_analytics.ucc_forecast_cbe
WHERE model = 'Adaptive_Ensemble'
  AND grain_level = 'total'
ORDER BY ds;

-- ITSG
SELECT ds, model, forecast, source, cumulative_qtd, boq_weekly,
       boq_cumulative_qtd, pace_vs_boq_pct, quarter
FROM datagroup_mdl.mdl_sales_analytics.itsg_forecast_cbe
WHERE model = 'Adaptive_Ensemble'
  AND grain_level = 'total'
ORDER BY ds;
```

### Pattern E - Full Quantile Fan (BOQ tables)
```sql
-- UCC (p10 to p90 by 10)
SELECT ds, model, p10, p20, p30, p40, p50, p60, p70, p80, p90, quarter
FROM datagroup_mdl.mdl_sales_analytics.ucc_forecast_boq
WHERE model = 'Adaptive_Ensemble'
  AND grain_level = 'total'
ORDER BY ds;

-- ITSG (p10/p50/p90 only)
SELECT ds, model, p10, p50, p90, quarter
FROM datagroup_mdl.mdl_sales_analytics.itsg_forecast_boq
WHERE model = 'Adaptive_Ensemble'
  AND grain_level = 'total'
ORDER BY ds;
```

### Pattern F - Model Accuracy Leaderboard
```sql
SELECT product, model, model_wape, model_mape
FROM datagroup_mdl.mdl_sales_analytics.forecast_results_latest
WHERE grain_level = 'total'
  AND forecast_type = 'rolling'
  AND forecast_view = 'adaptive'
  AND model != 'ACTUAL'
GROUP BY product, model, model_wape, model_mape
ORDER BY product, model_wape;
```

### Pattern G - AI Insights
```sql
SELECT product, momentum, risk_level, model_confidence, insights_json
FROM datagroup_mdl.mdl_sales_analytics.forecast_insights
WHERE run_date = (
  SELECT MAX(run_date)
  FROM datagroup_mdl.mdl_sales_analytics.forecast_insights
);
```

### Pattern H - Graph Explanations (per chart)
```sql
SELECT explanation, bullets
FROM datagroup_mdl.mdl_sales_analytics.forecast_graph_explanations
WHERE graph_key = :graph_key
  AND product = :product
  AND region = :region
  AND scope_period = :scope_period;
```

### Pattern I - Multi-Model Comparison
```sql
SELECT ds, model, p50, forecast_type
FROM datagroup_mdl.mdl_sales_analytics.forecast_results_latest
WHERE forecast_view = 'adaptive'
  AND product = :product
  AND grain_level = 'total'
  AND model != 'ACTUAL'
ORDER BY model, ds;
```

### Pattern J - Vintage Tracking (forecast drift)
```sql
SELECT ds, p50, run_date_utc, model
FROM datagroup_mdl.mdl_sales_analytics.ucc_forecast_vintages
WHERE model = 'Adaptive_Ensemble'
  AND grain_level = 'total'
ORDER BY run_date_utc, ds;
```

### Pattern K - By Geography
```sql
SELECT ds, sales_market, p50, forecast_type
FROM datagroup_mdl.mdl_sales_analytics.forecast_results_latest
WHERE forecast_view = 'adaptive'
  AND model = 'Adaptive_Ensemble'
  AND grain_level = 'market'
  AND product = :product
ORDER BY sales_market, ds;
```

## 5. Graph Specifications

| graph_key | Title | Data Source | X-Axis | Y-Axis | Series |
|---|---|---|---|---|---|
| weekly | Weekly Forecast vs Actuals | forecast_results_latest | ds | p50 + actual | forecast_type + p10/p90 band |
| ytd | Running Totals - YTD Cumulative | forecast_results_latest | ds | cumsum(actual + p50) | product |
| seasonality | Historical Seasonality by ISO Week | forecast_results_latest | iso_week | avg(actual) | year |
| by_product | Forecast by Product and Geography | forecast_results_latest | ds | p50 | product x sales_market |
| monthly | Monthly Actuals vs Forecast | forecast_results_latest | month | sum(actual), sum(p50) | model |
| accuracy | Model Accuracy Leaderboard | forecast_results_latest | model | model_wape | product |
| model_lab | Model Lab - Per-Model Forecast | forecast_results_latest | ds | p50 | model |
| backtest | Forecast vs Reality (Backtest) | ucc/itsg_forecast_vintages | ds | p50 vs actual | vintage date |
| quarterly_outlook | Quarterly Summary | forecast_results_latest + cbe | quarter | sum(p50) | actual vs forecast |
| multi_model | Multi-Model Forecast Comparison | forecast_results_latest | ds | p50 | model |

## 6. Advanced Panels

| Panel | Primary Table | Key Columns | Notes |
|---|---|---|---|
| 2027 Outlook | forecast_results_latest where forecast_type='forward_1yr' | p10, p50, p90 | Only models with extension data |
| Pace vs Plan | ucc_forecast_cbe / itsg_forecast_cbe | pace_vs_boq_pct, cumulative_qtd | source='actual' for closed and 'forecast' for open |
| BOQ Cumulative QTD | ucc_forecast_cbe / itsg_forecast_cbe | boq_cumulative_qtd, cumulative_qtd | Within-quarter running totals |
| Full Quantile Fan | ucc_forecast_boq, itsg_forecast_boq | UCC: p10..p90 by 10; ITSG: p10/p50/p90 | Product-specific quantile density |
| AI Insights | forecast_insights | insights_json, momentum, risk_level | Deterministic, no LLM at read time |
| Explain This Chart | forecast_graph_explanations | explanation, bullets | Keyed by graph/product/region/period |

## 7. Update Frequency

| Table | Frequency | Trigger |
|---|---|---|
| forecast_results | Daily ~07:00 ET | forecast_unify_publish job |
| forecast_insights | Daily ~07:00 ET | Same job |
| forecast_graph_explanations | Daily ~07:00 ET | Same job |
| ucc_forecast_v5 / boq / cbe | Daily | UCC Forecast Foundation V5 notebook |
| itsg_forecast_v5 / boq / cbe | Daily | ITSG Growth ARR Forecasting v5 notebook |
| ucc_forecast_vintages | Daily append | UCC notebook |
| monitoring baselines | Daily overwrite | Respective product notebooks |

## 8. Key Business Rules

- Fiscal year equals calendar year. FY2026 ends on 2026-12-28 (last Monday).
- Recommended model is Adaptive_Ensemble (`recommended_for_exec=1`).
- Quarter landing KPI equals `actuals_qtd + forecast_remaining` (rolling weeks).
- UCC quantiles upstream: p10 to p90 with increase of 10; app reads p10/p50/p90.
- ITSG quantiles upstream: p10/p50/p90 only; app reads p10/p50/p90.
- CBE source logic: `actual` for weeks up to last closed week, `forecast` for open weeks.
- `pace_vs_boq_pct`: positive means ahead of plan, negative means behind plan.
- Products: UCC, ITSG, Total.
- Markets: NA, EMEA, APAC, LATAM, Unknown, Total.
- Dollar values are weekly Growth ARR in USD (not annualized, not monthly).